In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp
import sys

In [3]:
spark = SparkSession.builder \
        .appName("FAOSTAT Data Ingestion") \
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
        .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog") \
        .config("spark.sql.catalog.lakehouse.catalog-impl", "org.apache.iceberg.jdbc.JdbcCatalog") \
        .config("spark.sql.catalog.lakehouse.uri", "jdbc:postgresql://localhost:5432/metastore") \
        .config("spark.sql.catalog.lakehouse.jdbc.user", "admin") \
        .config("spark.sql.catalog.lakehouse.jdbc.password", "password") \
        .config("spark.sql.catalog.lakehouse.warehouse", "s3a://lakehouse/warehouse") \
        .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
        .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
        .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
        .config("spark.hadoop.fs.s3a.path.style.access", "true") \
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
        .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3,org.apache.hadoop:hadoop-aws:3.3.4,org.postgresql:postgresql:42.6.0,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
        .getOrCreate()

:: loading settings :: url = jar:file:/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/lap16470/.ivy2/cache
The jars for the packages stored in: /Users/lap16470/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
org.postgresql#postgresql added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e2d26356-0360-422a-9fbf-b0550754b46d;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found org.postgresql#postgresql;42.6.0 in local-m2-cache
	found org.checkerframework#checker-qual;3.31.0 in local-m2-cache
:: resolution report :: resolve 1209ms :: artifacts dl 185ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	

In [4]:
spark.sql("show tables from lakehouse.bronze").show()

26/03/07 16:06:35 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


+---------+---------------+-----------+
|namespace|      tableName|isTemporary|
+---------+---------------+-----------+
|   bronze|faostat_qcl_raw|      false|
+---------+---------------+-----------+



In [5]:
spark.sql("select * from lakehouse.bronze.faostat_qcl_raw limit 1").show()

26/03/07 16:07:09 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+---------+---------------+-----------+---------+---------------+-----------------+------------+--------------+---------+----+----+-----+----+----+--------------------+
|Area Code|Area Code (M49)|       Area|Item Code|Item Code (CPC)|             Item|Element Code|       Element|Year Code|Year|Unit|Value|Flag|Note|      ingestion_time|
+---------+---------------+-----------+---------+---------------+-----------------+------------+--------------+---------+----+----+-----+----+----+--------------------+
|        2|           '004|Afghanistan|      221|         '01371|Almonds, in shell|        5312|Area harvested|     1961|1961|  ha|  0.0|   A|NULL|2026-03-07 16:01:...|
+---------+---------------+-----------+---------+---------------+-----------------+------------+--------------+---------+----+----+-----+----+----+--------------------+

